In [ ]:
from manim import *
import numpy as np
from scipy.signal import convolve2d

class CNNInsight(ThreeDScene):
    def construct(self):
        # ---------------------------------------------------------
        # 1. MATHEMATICAL SETUP
        # ---------------------------------------------------------
        # Input: 8x8 (Integers 1-9)
        input_data = np.random.randint(1, 10, (8, 8))
        
        # Conv: 8x8 -> 5x5. Kernel 4x4
        filter_kernel = np.random.randint(-1, 2, (4, 4))
        
        # Calculate Conv Result (Valid padding, Stride 1)
        conv_res = convolve2d(input_data, np.flipud(np.fliplr(filter_kernel)), mode='valid')
        
        # Pooling: 5x5 -> 3x3. Stride=1, Kernel=3.
        pool_res = np.zeros((3, 3))
        for r in range(3):
            for c in range(3):
                pool_res[r,c] = np.max(conv_res[r:r+3, c:c+3])
        
        # Softmax (9 inputs)
        flattened = pool_res.flatten()
        dense_weights = np.random.randn(9, 9)
        logits = flattened @ dense_weights
        exps = np.exp(logits - np.max(logits))
        softmax = exps / np.sum(exps)

        # ---------------------------------------------------------
        # 2. HELPER FUNCTION
        # ---------------------------------------------------------
        def create_yz_grid(data, color, label_text, x_pos, hide_text=False):
            rows, cols = data.shape
            g = VGroup()
            cells = []
            texts = []
            
            for r in range(rows):
                row_cells = []
                row_txts = []
                for c in range(cols):
                    # Square cell
                    sq = Square(side_length=0.5)
                    sq.set_stroke(color, 2)
                    sq.set_fill(color, 0.1)
                    
                    # Position in YZ plane
                    y = (rows/2 - r - 0.5) * 0.5
                    z = (c - cols/2 + 0.5) * 0.5
                    
                    sq.rotate(PI/2, axis=UP) # Face X-axis
                    sq.move_to([x_pos, y, z])
                    
                    # Text value
                    val = data[r,c]
                    s_val = f"{int(val)}" if isinstance(val, (int, np.int64, np.int32)) else f"{val:.1f}"
                    t = Text(s_val, font_size=14, color=WHITE)
                    t.rotate(PI/2, axis=UP).rotate(PI/2, axis=RIGHT)
                    t.move_to(sq)
                    
                    if hide_text:
                        t.set_opacity(0)
                        
                    g.add(sq, t)
                    row_cells.append(sq)
                    row_txts.append(t)
                cells.append(row_cells)
                texts.append(row_txts)
            
            lbl = Text(label_text, font_size=24, color=color)
            lbl.rotate(PI/2, axis=UP).rotate(PI/2, axis=RIGHT)
            lbl.next_to(g, UP, buff=0.5)
            
            return g, cells, texts, lbl

        # ---------------------------------------------------------
        # 3. SCENE BUILD
        # ---------------------------------------------------------
        # Initial Camera
        self.set_camera_orientation(phi=75 * DEGREES, theta=45 * DEGREES, zoom=0.6)

        # Create Layers (Initialize Output text as hidden)
        in_g, in_cells, _, in_lbl = create_yz_grid(input_data, BLUE, "Input", x_pos=-4)
        conv_g, conv_cells, conv_txts, conv_lbl = create_yz_grid(conv_res, YELLOW, "Conv", x_pos=-1, hide_text=True)
        pool_g, pool_cells, pool_txts, pool_lbl = create_yz_grid(pool_res, RED, "Pool", x_pos=2, hide_text=True)
        dense_g, dense_cells, dense_txts, dense_lbl = create_yz_grid(np.zeros((9,1)), GREEN, "Softmax", x_pos=5, hide_text=True)

        # Filter Object
        filt_g, _, _, _ = create_yz_grid(filter_kernel, ORANGE, "K", x_pos=-3.5)
        
        # Grouping
        input_layer = VGroup(in_g, in_lbl)
        conv_layer = VGroup(conv_g, conv_lbl)
        pool_layer = VGroup(pool_g, pool_lbl)
        dense_layer = VGroup(dense_g, dense_lbl)
        filter_layer = VGroup(filt_g)
        filter_layer.set_opacity(0) # Hide initially

        self.play(
            Create(input_layer),
            Create(conv_layer),
            Create(pool_layer),
            Create(dense_layer),
            run_time=1.5
        )
        self.wait(1)

        # ---------------------------------------------------------
        # 4. CONVOLUTION
        # ---------------------------------------------------------
        
        # 1. Move Camera to Side View & Zoom In
        self.move_camera(
            phi=90 * DEGREES, theta=0 * DEGREES, 
            zoom=1.0, 
            run_time=1.5
        )

        # 2. Vanish irrelevant layers + Show Filter
        self.play(
            FadeOut(pool_layer),
            FadeOut(dense_layer),
            FadeIn(filter_layer),
            run_time=1
        )

        # 3. Relayout: Input Left, Conv Right
        # Note: In this camera view (phi=90, theta=0), Y-axis moves Left/Right.
        # We put Input at Y=-3 (Left) and Conv at Y=3 (Right)
        self.play(
            input_layer.animate.move_to([-4, -3, 0]),  # Left
            conv_layer.animate.move_to([-1, 3, 0]),    # Right
            filter_layer.animate.set_opacity(1),
            run_time=1.5
        )

        # 4. HUD
        hud_title = Text("Operation: Convolution", font_size=36).to_edge(UP)
        hud_dims_in = Text("Input: 8x8", font_size=24, color=BLUE).to_corner(UL)
        hud_dims_filt = Text("Filter: 4x4", font_size=24, color=ORANGE).next_to(hud_dims_in, DOWN, aligned_edge=LEFT)
        hud_dims_out = Text("Output: 5x5", font_size=24, color=YELLOW).to_corner(UR)
        
        self.add_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_filt, hud_dims_out)
        self.play(Write(hud_title), FadeIn(hud_dims_in), FadeIn(hud_dims_filt), FadeIn(hud_dims_out))

        # 5. Highlights (FIXED: Using explicit Rectangle instead of SurroundingRectangle)
        p1 = in_cells[0][0].get_center()
        p2 = in_cells[3][3].get_center()
        # Calculate width/height based on Z (width) and Y (height) differences
        h_width = abs(p1[2]-p2[2]) + 0.5
        h_height = abs(p1[1]-p2[1]) + 0.5
        
        in_highlight = Rectangle(width=h_width, height=h_height, color=ORANGE)
        in_highlight.rotate(PI/2, axis=UP) # Align with YZ plane
        in_highlight.move_to((p1+p2)/2)

        self.play(Create(in_highlight))

        # 6. Animation Loop
        count = 0
        for r in range(5):
            for c in range(5):
                # Targets
                top_left = in_cells[r][c].get_center()
                bot_right = in_cells[r+3][c+3].get_center()
                target_in = (top_left + bot_right) / 2
                
                target_out = conv_cells[r][c].get_center()
                
                rt = 0.5 if count < 2 else 0.1

                # Move Highlight & Filter
                self.play(
                    in_highlight.animate.move_to(target_in),
                    filt_g.animate.move_to(target_in + RIGHT * 0.5),
                    run_time=rt
                )

                # Show Output Flying
                val = conv_res[r,c]
                flying_text = Text(f"{int(val)}", font_size=20, color=YELLOW)
                flying_text.rotate(PI/2, UP).rotate(PI/2, RIGHT)
                flying_text.move_to(target_in)
                
                self.play(flying_text.animate.move_to(target_out), run_time=rt)
                
                self.remove(flying_text)
                conv_txts[r][c].set_opacity(1)
                self.add(conv_txts[r][c])
                
                count += 1

        # 7. Cleanup
        self.play(FadeOut(in_highlight), FadeOut(filter_layer))
        self.remove_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_filt, hud_dims_out)
        self.remove(hud_title, hud_dims_in, hud_dims_filt, hud_dims_out)

        # 8. Reset Positions
        self.play(
            input_layer.animate.move_to([-4, 0, 0]),
            conv_layer.animate.move_to([-1, 0, 0]),
            run_time=1
        )

        # 9. Transition Out
        self.move_camera(phi=75 * DEGREES, theta=45 * DEGREES, zoom=0.6, run_time=1.5)
        self.play(FadeIn(pool_layer), FadeIn(dense_layer), run_time=1)
        self.wait(1)

        # ---------------------------------------------------------
        # 5. MAX POOLING
        # ---------------------------------------------------------
        
        # 1. Move Camera & Zoom In
        self.move_camera(phi=90 * DEGREES, theta=0 * DEGREES, zoom=1.1, run_time=1.5)
        
        # 2. Vanish & Relayout
        self.play(
            FadeOut(input_layer),
            FadeOut(dense_layer),
            run_time=1
        )
        
        # Conv (Input) -> Left (-3), Pool (Output) -> Right (3)
        self.play(
            conv_layer.animate.move_to([-1, -3, 0]), 
            pool_layer.animate.move_to([2, 3, 0]), 
            run_time=1.5
        )

        # 3. HUD
        hud_title = Text("Operation: Max Pooling", font_size=36, color=RED).to_edge(UP)
        hud_dims_in = Text("Input: 5x5", font_size=24, color=YELLOW).to_corner(UL)
        hud_dims_kernel = Text("Kernel: 3x3", font_size=24, color=RED).next_to(hud_dims_in, DOWN, aligned_edge=LEFT)
        hud_dims_out = Text("Output: 3x3", font_size=24, color=RED).to_corner(UR)
        
        self.add_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_kernel, hud_dims_out)
        self.play(Write(hud_title), FadeIn(hud_dims_in), FadeIn(hud_dims_kernel), FadeIn(hud_dims_out))

        # 4. Highlights (FIXED: Using explicit Rectangle)
        c1 = conv_cells[0][0].get_center()
        c2 = conv_cells[2][2].get_center()
        
        # Recalculate dimensions based on current positions
        p_width = abs(c1[2]-c2[2]) + 0.5
        p_height = abs(c1[1]-c2[1]) + 0.5
        
        pool_src_highlight = Rectangle(width=p_width, height=p_height, color=RED)
        pool_src_highlight.rotate(PI/2, axis=UP)
        pool_src_highlight.move_to((c1+c2)/2)
        
        self.play(Create(pool_src_highlight))

        # 5. Animation Loop
        count = 0
        for r in range(3):
            for c in range(3):
                top_left = conv_cells[r][c].get_center()
                bot_right = conv_cells[r+2][c+2].get_center()
                target_src = (top_left + bot_right) / 2
                target_dest = pool_cells[r][c].get_center()
                
                rt = 0.5 if count < 2 else 0.15
                
                self.play(pool_src_highlight.animate.move_to(target_src), run_time=rt)
                
                val = pool_res[r,c]
                flying_text = Text(f"{int(val)}", font_size=20, color=RED)
                flying_text.rotate(PI/2, UP).rotate(PI/2, RIGHT)
                flying_text.move_to(target_src)
                
                self.play(flying_text.animate.move_to(target_dest), run_time=rt)
                
                self.remove(flying_text)
                pool_txts[r][c].set_opacity(1)
                self.add(pool_txts[r][c])
                count += 1
        
        # 6. Cleanup
        self.play(FadeOut(pool_src_highlight))
        self.remove_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_kernel, hud_dims_out)
        self.remove(hud_title, hud_dims_in, hud_dims_kernel, hud_dims_out)

        # 7. Reset Positions
        self.play(
            conv_layer.animate.move_to([-1, 0, 0]),
            pool_layer.animate.move_to([2, 0, 0]),
            run_time=1
        )

        # 8. Transition Out
        self.move_camera(phi=75 * DEGREES, theta=45 * DEGREES, zoom=0.6, run_time=1.5)
        self.play(FadeIn(input_layer), FadeIn(dense_layer), run_time=1)
        self.wait(1)

        # ---------------------------------------------------------
        # 6. FLATTEN & SOFTMAX
        # ---------------------------------------------------------
        
        # 1. Move Camera & Zoom In
        self.move_camera(phi=90 * DEGREES, theta=0 * DEGREES, zoom=1.2, run_time=1.5)
        
        # 2. Vanish & Relayout
        self.play(
            FadeOut(input_layer),
            FadeOut(conv_layer),
            run_time=1
        )
        
        # Pool (Input) -> Left, Dense (Output) -> Right
        self.play(
            pool_layer.animate.move_to([2, -3, 0]),   # Left
            dense_layer.animate.move_to([5, 3, 0]),   # Right
            run_time=1.5
        )

        # 3. HUD
        hud_title = Text("Operation: Flatten & Softmax", font_size=32, color=GREEN).to_edge(UP)
        hud_dims_in = Text("Input: 3x3", font_size=24, color=RED).to_corner(UL)
        hud_dims_out = Text("Output: 9x1", font_size=24, color=GREEN).to_corner(UR)
        self.add_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_out)
        self.play(Write(hud_title), FadeIn(hud_dims_in), FadeIn(hud_dims_out))
        
        # 4. Flatten Animation
        for r in range(3):
             for c in range(3):
                flat_idx = r * 3 + c
                end = dense_cells[flat_idx][0].get_center()
                
                flying_obj = pool_cells[r][c].copy()
                flying_obj.set_color(GREEN)
                
                self.play(flying_obj.animate.move_to(end), run_time=0.1)
                self.remove(flying_obj)
                dense_txts[flat_idx][0].set_opacity(1)
        
        # 5. Softmax Update
        self.wait(0.5)
        sm_anims = []
        for i in range(9):
            val = softmax[i]
            t_soft = Text(f"{val:.2f}", font_size=12, color=GREEN)
            t_soft.rotate(PI/2, UP).rotate(PI/2, RIGHT).move_to(dense_cells[i][0])
            sm_anims.append(Transform(dense_txts[i][0], t_soft))
            
        self.play(*sm_anims, run_time=1.0)

        # 6. Cleanup & Reset
        self.remove_fixed_in_frame_mobjects(hud_title, hud_dims_in, hud_dims_out)
        self.remove(hud_title, hud_dims_in, hud_dims_out)
        
        self.play(
            pool_layer.animate.move_to([2, 0, 0]),
            dense_layer.animate.move_to([5, 0, 0]),
            run_time=1
        )

        # 7. Final View
        self.move_camera(
            phi=75 * DEGREES, theta=45 * DEGREES, 
            zoom=0.6, focal_point=ORIGIN, 
            run_time=2.0
        )
        self.play(
            FadeIn(input_layer),
            FadeIn(conv_layer),
            run_time=1
        )
        self.wait(2)


%manim -qk -v warning CNNInsight

Manim Community v0.19.0